In [1]:
from src.acquisition import *

from pathlib import Path
root = Path.cwd()

import pandas as pd

from osgeo import gdal
gdal.SetConfigOption("GDAL_HTTP_UNSAFESSL", "YES")
gdal.SetConfigOption('GDAL_HTTP_COOKIEFILE','~/cookies.txt')
gdal.SetConfigOption('GDAL_HTTP_COOKIEJAR', '~/cookies.txt')
gdal.SetConfigOption('GDAL_DISABLE_READDIR_ON_OPEN','YES')
gdal.SetConfigOption('CPL_VSIL_CURL_ALLOWED_EXTENSIONS','TIF')

### Query API

In [22]:
TC_FILENAME = 'tc_baltimore_buffer_30m.tif'
CITY = 'baltimore'
BOUNDARY_FILE = 'baltimore_boundary.gpkg'
FROM_DATE = '2021-04-01'
TO_DATE = '2021-11-30'

landsat_catalog = "HLSL30.v2.0"
landsat_bands = ["B02","B03","B04","B05","B06","B07","B10","B11","Fmask"]
sentinel_catalog = "HLSS30.v2.0"
sentinel_bands = ["B02","B03","B04","B05","B06","B07","B8A","B11","B12","Fmask"]

# read in city vector data
boundary, bbox_utm, bbox_4326, shoreline = read_vector(boundary_file=BOUNDARY_FILE,root=root,cheasapeake=True)

# query API, crop satellite data to boundaries
# sentinel_crop = get_stac_items(catalog_name=sentinel_catalog,
#                    from_date=FROM_DATE, 
#                    to_date=TO_DATE,
#                    bbox_utm=bbox_utm,
#                    bbox_4326=bbox_4326,
#                    boundary=boundary,
#                    bands=sentinel_bands,
#                    shoreline=shoreline)

# FROM_DATE = '2020-04-01'
# TO_DATE = '2020-11-30'
landsat_crop = get_stac_items(catalog_name=landsat_catalog,
                   from_date=FROM_DATE, 
                   to_date=TO_DATE,
                   bbox_utm=bbox_utm,
                   bbox_4326=bbox_4326,
                   boundary=boundary,
                   bands=landsat_bands,
                   shoreline=shoreline)



In [ ]:
l_drop_2019 = check_data(landsat_crop,'first')
l_drop_2021 = check_data(landsat_crop,'first')

In [ ]:
l_sept = l_drop_2021.isel(time=4)
l_sept = l_sept.expand_dims({'time':[l_sept.coords['time'].values]})

In [44]:
l_june = l_drop_2021.isel(time=2)
l_june = l_june.expand_dims({'time':[l_june.coords['time'].values]})

In [ ]:
s_drop_2019 = check_data(sentinel_crop,'first')

In [ ]:
s_drop_2018 = check_data(sentinel_crop,'first')
s_may = s_drop_2018.isel(time=slice(1,3))

In [33]:
# save image dates to csv
all_landsat_times = np.concat([l_drop_2019.time.values, l_sept.time.values,l_june.time.values])
all_sentinel_times = np.concat([s_drop_2019.time.values,s_may.time.values])

sdf = pd.DataFrame({'sentinel':all_sentinel_times})
ldf = pd.DataFrame({'landsat':all_landsat_times})       

new = pd.concat([sdf, ldf], axis=1) 
new.to_csv(root / 'output' / CITY / f'{CITY}_satellite_images.csv')

In [ ]:
# change landsat dates to same year and combine

# june and sept from 2021
l_june['time'] = l_june.time.to_index() - pd.Timedelta(days=365*2+3)
l_sept['time'] = l_sept.time.to_index() - pd.Timedelta(days=365*2)

# may from 2018
s_may['time'] = s_may.time.to_index() + pd.Timedelta(days=365)


In [ ]:
l_combine = l_drop_2019.combine_first(l_june).combine_first(l_sept)

s_combine = s_drop_2019.combine_first(s_may)

### Cloud Mask and Scale

In [50]:
l_masked = mask_and_scale2(l_combine, sensor='landsat',scale_factor1= .0001, scale_factor2= .01)
s_masked = mask_and_scale2(s_combine,sensor='sentinel',scale_factor1= .0001, scale_factor2=None)

### Calculate vegetation indices, merge landsat and sentinel data

In [53]:
all_unique_indices, all_annual_unique_indices = get_unique_indices(l_data=l_masked,s_data=s_masked)
all_shared_indices, all_annual_shared_indices = get_shared_indices(l_data=l_masked,s_data=s_masked)

t = concatenate_and_save_data(annual1=all_annual_unique_indices,annual2=all_annual_shared_indices,
                              monthly1=all_unique_indices,monthly2=all_shared_indices,root=root,filename=f'{CITY}_hls_indices2')

c:\Users\roseh\miniconda3\envs\rs-env\Lib\site-packages\dask\array\core.py:4888: PerformanceWarning: Increasing number of chunks by factor of 11
  result = blockwise(
c:\Users\roseh\miniconda3\envs\rs-env\Lib\site-packages\dask\array\core.py:4888: PerformanceWarning: Increasing number of chunks by factor of 11
  result = blockwise(


In [55]:
all_unique_bands, all_annual_unique = get_unique_bands(l_data=l_masked,s_data=s_masked)
all_shared_bands, all_annual_shared = get_shared_bands(landsat_data=l_masked,sentinel_data=s_masked)


s = concatenate_and_save_data(annual1=all_annual_shared,annual2=all_annual_unique,
                              monthly1=all_unique_bands,monthly2=all_shared_bands, root=root,filename=f'{CITY}_hls_bands2',
                              # extra arguments needed for adding tree canopy layer
                              add_canopy=True,shoreline=shoreline,tc_filename=TC_FILENAME,city_boundary=boundary)


c:\Users\roseh\miniconda3\envs\rs-env\Lib\site-packages\dask\utils.py:78: RuntimeWarning: All-NaN slice encountered
  return func(*args, **kwargs)


### Save as Rasters

In [ ]:
# def save_monthly_rasters(data,filename):
#     months = ['april','may','june','july','august','september','october','november']
#     for i in range(0,8):
#         data.isel(time=i).rio.to_raster(root / 'data' / f'{filename}_{months[i]}.tif')


# all_annual.isel(time=0).rio.to_raster(root / 'data' / 'annual_bands.tif')     

# save_monthly_rasters(all_shared_bands,filename='shared_bands')
# save_monthly_rasters(all_unique_bands,filename='unique_bands')
# save_monthly_rasters(all_shared_indices,filename='shared_indices')